# shuttle-c2-vs-c4: current SVM/KNN/RF/LR comparison

Methods: no resampling, SMOTE, BSMOTE, ROS, ADASYN, CASB, CASB-relaxed, CASB-edited, CASB-ROS. RUS, SMOTEENN and MCSMOTE are excluded. Historical records remain in metadata.

CASB uses its original interpolation generator; CASB-ROS duplicates sources selected with CASB's original 50/30/20 quotas. This modular extension is inspired by the UGO paper, not a reproduction of UGO. Five supplied folds, seeds 42/137, same training-only ratio search (0.5/0.75/1) maximizing (G-mean+F1)/2. RBF SVM C=1 gamma=scale; distance-weighted KNN k=min(7,resampled count). RF: 100 trees, minimum leaf size 2; LR: C=1, max_iter=3000. These classifiers were retained after inspecting benchmark results, so the expanded comparison is exploratory. All preprocessing is fitted on training data. Abalone uses one-hot encoding.

Run all cells to inspect cached evidence. RF/LR evidence is imported from progress.md; GaussianNB remains in that report. Use the shared data_splits, choose_ratio and evaluate_fit functions in binary_oversampling.py to refit any saved setting. ADASYN failures are reported without fallback; incomplete method averages are withheld. All datasets have been inspected during development; this is exploratory evidence.


In [1]:
import sys, os
from pathlib import Path
sys.dont_write_bytecode=True
sys.path.insert(0,str(Path(os.environ.get('TEMP','/tmp'))/'ml_project_runtime'))
sys.path.insert(0,str(Path.cwd()))
import pandas as pd
from IPython.display import display, Markdown
from binary_oversampling import current_dataset_records, CURRENT_METHODS
import json
nb=json.loads(Path('29_shuttle-c2-vs-c4_CASB.ipynb').read_text(encoding='utf8'))
records=nb['metadata']['current_comparison']['records']
classifiers=['svm', 'knn', 'rf', 'lr']
frame=pd.DataFrame(records)
assert len(frame)==360
if 'error' not in frame:frame['error']=None
errors=frame[frame.error.notna()]
print('Attempted:',len(frame),'Failed:',len(errors))
if len(errors):display(errors[['classifier','method','seed','fold','error']])
frame['composite']=(frame.gmean+frame.f1)/2
metrics=['gmean','f1','precision','recall','specificity','ba','ap','auc','composite']


Attempted: 360 Failed: 12


,classifier,method,seed,fold,error
4,svm,adasyn,42,1,Not any neigbours belong to the majority class...
13,knn,adasyn,42,1,Not any neigbours belong to the majority class...
94,svm,adasyn,137,1,Not any neigbours belong to the majority class...
103,knn,adasyn,137,1,Not any neigbours belong to the majority class...
148,svm,adasyn,137,4,Not any neigbours belong to the majority class...
157,knn,adasyn,137,4,Not any neigbours belong to the majority class...
184,rf,adasyn,42,1,RuntimeError: Not any neigbours belong to the ...
193,lr,adasyn,42,1,RuntimeError: Not any neigbours belong to the ...
274,rf,adasyn,137,1,RuntimeError: Not any neigbours belong to the ...
283,lr,adasyn,137,1,RuntimeError: Not any neigbours belong to the ...


In [2]:
ok=frame[frame.error.isna()]
counts=ok.groupby(['classifier','method']).size()
summary=ok.groupby(['classifier','method'])[metrics].mean()
for key in summary.index:
    if counts.loc[key]!=10:summary.loc[key,:]=float('nan')
summary=summary.reindex(pd.MultiIndex.from_product([classifiers,CURRENT_METHODS],names=['classifier','method']))
for clf in classifiers:
    display(Markdown('## '+clf.upper()+': averages (%)'))
    display((100*summary.loc[clf]).round(3))
combined=summary.groupby('method').mean().reindex(CURRENT_METHODS)
for method in CURRENT_METHODS:
    if summary.xs(method,level='method').isna().any().any():combined.loc[method,:]=float('nan')
display(Markdown('## Equal average of SVM, KNN, RF and LR (%)'))
display((100*combined).round(3))
print('NaN indicates incomplete coverage, not a zero score.')


## SVM: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738
smote,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738
bsmote,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738
ros,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738
casb_relaxed,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738
casb_edited,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738
casb_ros,94.142,93.333,100.0,90.0,100.0,95.0,100.0,100.0,93.738


## KNN: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738
smote,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738
bsmote,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738
ros,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738
casb_relaxed,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738
casb_edited,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738
casb_ros,94.142,93.333,100.0,90.0,100.0,95.0,90.769,95.0,93.738


## RF: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,97.071,96.667,100.0,95.0,100.0,97.5,100.0,100.0,96.869
smote,100.000,100.000,100.0,100.0,100.0,100.0,100.0,100.0,100.000
bsmote,100.000,100.000,100.0,100.0,100.0,100.0,100.0,100.0,100.000
ros,100.000,100.000,100.0,100.0,100.0,100.0,100.0,100.0,100.000
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,100.000,100.000,100.0,100.0,100.0,100.0,100.0,100.0,100.000
casb_relaxed,100.000,100.000,100.0,100.0,100.0,100.0,100.0,100.0,100.000
casb_edited,100.000,100.000,100.0,100.0,100.0,100.0,100.0,100.0,100.000
casb_ros,100.000,100.000,100.0,100.0,100.0,100.0,100.0,100.0,100.000


## LR: averages (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202
smote,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202
bsmote,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202
ros,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202
casb_relaxed,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202
casb_edited,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202
casb_ros,93.738,86.667,90.0,90.0,99.2,94.6,90.0,99.2,90.202


## Equal average of SVM, KNN, RF and LR (%)

,gmean,f1,precision,recall,specificity,ba,ap,auc,composite
method,,,,,,,,,
original,94.773,92.500,97.5,91.25,99.8,95.525,95.192,98.55,93.637
smote,95.506,93.333,97.5,92.50,99.8,96.150,95.192,98.55,94.419
bsmote,95.506,93.333,97.5,92.50,99.8,96.150,95.192,98.55,94.419
ros,95.506,93.333,97.5,92.50,99.8,96.150,95.192,98.55,94.419
adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
casb,95.506,93.333,97.5,92.50,99.8,96.150,95.192,98.55,94.419
casb_relaxed,95.506,93.333,97.5,92.50,99.8,96.150,95.192,98.55,94.419
casb_edited,95.506,93.333,97.5,92.50,99.8,96.150,95.192,98.55,94.419
casb_ros,95.506,93.333,97.5,92.50,99.8,96.150,95.192,98.55,94.419


NaN indicates incomplete coverage, not a zero score.


In [3]:
core=['smote','bsmote','ros','adasyn']
family=['casb','casb_relaxed','casb_edited','casb_ros']
for view,table in [(clf,summary.loc[clf]) for clf in classifiers]+[('equal four-classifier average',combined)]:
    output=[]
    for metric in ['gmean','f1','precision','recall']:
        values=table.loc[core+family,metric]
        if values.isna().any():
            output.append(dict(metric=metric,result='Incomplete coverage'));continue
        ours=values.loc[family];base=values.loc[core];delta=ours.max()-base.max()
        output.append(dict(metric=metric,best_variant=ours.idxmax(),family_percent=100*ours.max(),baseline=base.idxmax(),baseline_percent=100*base.max(),difference_pp=100*delta,result='WIN' if delta>1e-12 else 'LOSS' if delta < -1e-12 else 'TIE'))
    display(Markdown('## '+view+': per-metric family comparison'))
    display(pd.DataFrame(output).round(3))
print('Retrospective per-metric family maxima; not one deployable winning method. No composite winner.')
display(Markdown('## Fold-level evidence'))
with pd.option_context('display.max_rows',400):
    display(frame[['seed','fold','classifier','method','ratio']+metrics+['error']])


## svm: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


## knn: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


## rf: per-metric family comparison

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


... (3 outputs left out)

,metric,result
0,gmean,Incomplete coverage
1,f1,Incomplete coverage
2,precision,Incomplete coverage
3,recall,Incomplete coverage


Retrospective per-metric family maxima; not one deployable winning method. No composite winner.


## Fold-level evidence

,seed,fold,classifier,method,ratio,gmean,f1,precision,recall,specificity,ba,ap,auc,composite,error
0,42,1,svm,original,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
1,42,1,svm,smote,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
2,42,1,svm,bsmote,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
3,42,1,svm,ros,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
4,42,1,svm,adasyn,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,Not any neigbours belong to the majority class...
5,42,1,svm,casb,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
6,42,1,svm,casb_relaxed,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
7,42,1,svm,casb_edited,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
8,42,1,svm,casb_ros,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,1.000000,1.00,0.686887,NaN
9,42,1,knn,original,0.5,0.707107,0.666667,1.0,0.5,1.00,0.75,0.538462,0.75,0.686887,NaN
